# Обучение YOLO 11 для детекции углов шахматной доски (Keypoint Detection)Этот notebook содержит полный процесс обучения модели YOLO 11s для детекции 4 углов игрового поля шахматной доски.## Особенности:- Использование YOLO 11s (small) для keypoint detection- Детекция 4 углов игрового поля (a1, h1, h8, a8)- Датасет: chess-boards (1600x1200, с аугментациями)- Обучение в Google Colab## Формат данных:- Класс: Chess-Board (class_id=0)- Keypoints: 4 точки в формате YOLO (x1, y1, v1, x2, y2, v2, x3, y3, v3, x4, y4, v4)  - **visibility (v)** - флаг видимости точки:    - `0` = точка не видна (закрыта/не видна)    - `1` = точка видна, но за пределами изображения    - `2` = точка видна и находится в пределах изображения ← **используем это**  - Формат лейблов: class_id x1 y1 v1 x2 y2 v2 x3 y3 v3 x4 y4 v4 (13 чисел), YOLO требует x, y, visibility (12 чисел)  - Блокнот автоматически конвертирует формат, добавляя visibility=2.0- Формат: class_id + 4 точки с visibility (БЕЗ bbox)## Размер изображения:- Оригинальные изображения: 1600x1200- Для обучения используем 1280x1280 (квадрат, кратный 32, сохраняет детали)- Можно увеличить до 1600x1600, но потребуется больше GPU памяти и уменьшить batch size

## Установка зависимостей

In [1]:
# Установка зависимостей
!pip install ultralytics
!pip install supervision
!pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 13.2 MB/s eta 0:00:00


## Импорты

In [2]:
import os
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import yaml
from typing import List, Tuple
import json
import zipfile
import shutil
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Настройка путей и параметров

In [3]:
# Настройки проекта
PROJECT_DIR = Path('/content/chess_board_corners_yolo11')
DATASET_DIR = PROJECT_DIR / 'dataset'
MODELS_DIR = PROJECT_DIR / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'

# Создание директорий
for dir_path in [PROJECT_DIR, DATASET_DIR, MODELS_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Параметры обучения
MODEL_SIZE = 's'  # s = small (YOLO11s)
# Размер изображения для обучения (YOLO требует кратный 32)
# Можно использовать больший размер для лучшей точности (1280, 1600)
# Но учтите: больше размер = больше памяти GPU и медленнее обучение
IMG_SIZE = 1280  # Используем 1280x1280 для сохранения деталей (оригинал 1600x1200)
EPOCHS = 100
BATCH_SIZE = 8  # Уменьшаем batch для большого размера изображения
PATIENCE = 50  # Early stopping patience

# Keypoint detection параметры
NUM_KEYPOINTS = 4  # 4 угла доски
KEYPOINT_NAMES = ['corner_a1', 'corner_h1', 'corner_h8', 'corner_a8']

print(f"Проект создан: {PROJECT_DIR}")
print(f"Размер модели: YOLO11{MODEL_SIZE}")
print(f"Количество keypoints: {NUM_KEYPOINTS}")
print(f"Keypoints: {KEYPOINT_NAMES}")

Проект создан: /content/chess_board_corners_yolo11
Размер модели: YOLO11s
Количество keypoints: 4
Keypoints: ['corner_a1', 'corner_h1', 'corner_h8', 'corner_a8']


## Загрузка и подготовка датасета

In [4]:
# Обработка архива с датасетом
ARCHIVE_PATH = None

# Автоматический поиск архива в /content
if ARCHIVE_PATH is None:
    content_dir = Path("/content")
    zip_files = list(content_dir.glob("*.zip"))
    if zip_files:
        ARCHIVE_PATH = str(zip_files[0])
        print(f"Найден архив: {ARCHIVE_PATH}")
    else:
        print("⚠ Архив не найден. Загрузите zip-файл с датасетом chess-boards в /content/")
        print("Или укажите путь: ARCHIVE_PATH = '/content/chess-boards.zip'")

if ARCHIVE_PATH and Path(ARCHIVE_PATH).exists():
    print(f"Обработка: {ARCHIVE_PATH}")

    temp_extract = DATASET_DIR.parent / "temp_extract"
    temp_extract.mkdir(exist_ok=True)

    print("Распаковка...")
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(temp_extract)

    # Поиск структуры датасета
    extracted_dirs = list(temp_extract.rglob("train"))
    if not extracted_dirs:
        extracted_dirs = list(temp_extract.rglob("*train*"))
    if not extracted_dirs:
        # Пробуем найти chess-boards директорию
        chess_boards_dirs = list(temp_extract.rglob("chess-boards"))
        if chess_boards_dirs:
            dataset_root = chess_boards_dirs[0]
        else:
            dataset_root = temp_extract
    else:
        dataset_root = extracted_dirs[0].parent

    print(f"Найден датасет: {dataset_root}")

    # Создаем структуру
    for split in ['train', 'valid', 'test']:
        (DATASET_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
        (DATASET_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

    # Копируем файлы
    for split in ['train', 'valid', 'test']:
        split_dir = dataset_root / split
        if split_dir.exists():
            if (split_dir / 'images').exists():
                shutil.copytree(split_dir / 'images', DATASET_DIR / split / 'images', dirs_exist_ok=True)
            if (split_dir / 'labels').exists():
                shutil.copytree(split_dir / 'labels', DATASET_DIR / split / 'labels', dirs_exist_ok=True)

    # Копируем YAML файлы
    for yaml_file in list(dataset_root.glob("*.yaml")) + list(dataset_root.glob("*.yml")):
        shutil.copy(yaml_file, DATASET_DIR / yaml_file.name)

    shutil.rmtree(temp_extract)
    print(f"✓ Датасет обработан: {DATASET_DIR}")

    # Проверяем структуру
    print(f"\nСтруктура датасета:")
    for split in ['train', 'valid', 'test']:
        images_count = len(list((DATASET_DIR / split / 'images').glob('*'))) if (DATASET_DIR / split / 'images').exists() else 0
        labels_count = len(list((DATASET_DIR / split / 'labels').glob('*'))) if (DATASET_DIR / split / 'labels').exists() else 0
        print(f"  {split}: {images_count} images, {labels_count} labels")
else:
    print("❌ Датасет не найден. Загрузите архив и запустите ячейку выше")

Найден архив: /content/chess-boards.zip
Обработка: /content/chess-boards.zip
Распаковка...
Найден датасет: /content/chess_board_corners_yolo11/temp_extract/chess-boards
✓ Датасет обработан: /content/chess_board_corners_yolo11/dataset

Структура датасета:
  train: 294 images, 882 labels
  valid: 12 images, 36 labels
  test: 12 images, 36 labels


## Создание конфигурационного файла для keypoint detection

In [5]:
# Создание data.yaml для keypoint detection
def create_keypoint_yaml(dataset_dir: Path, num_keypoints: int, keypoint_names: List[str]) -> Path:
    """Создание YAML конфигурации для keypoint detection"""

    # Используем абсолютные пути для Colab
    train_path = str((dataset_dir / 'train' / 'images').resolve())
    val_path = str((dataset_dir / 'valid' / 'images').resolve()) if (dataset_dir / 'valid').exists() else str((dataset_dir / 'train' / 'images').resolve())
    test_path = str((dataset_dir / 'test' / 'images').resolve()) if (dataset_dir / 'test').exists() else str((dataset_dir / 'train' / 'images').resolve())

    config = {
        'path': str(dataset_dir.resolve()),
        'train': train_path,
        'val': val_path,
        'test': test_path,

        # Классы (для keypoint detection нужен хотя бы один класс)
        'nc': 1,  # 1 класс: Chess-Board
        'names': ['Chess-Board'],

        # Keypoint detection параметры
        'kpt_shape': [num_keypoints, 3],  # [num_keypoints, 3] где 3 = (x, y, visibility)
        'flip_idx': list(range(num_keypoints)),  # Индексы для flip augmentation (пока без flip)
    }

    yaml_path = dataset_dir / 'data.yaml'
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

    print(f"✓ Конфигурация создана: {yaml_path}")
    print(f"\nКонфигурация:")
    print(yaml.dump(config, default_flow_style=False, allow_unicode=True))

    return yaml_path

# Создаем конфигурацию
dataset_yaml = create_keypoint_yaml(DATASET_DIR, NUM_KEYPOINTS, KEYPOINT_NAMES)
print(f"\n✓ YAML файл готов для keypoint detection")

✓ Конфигурация создана: /content/chess_board_corners_yolo11/dataset/data.yaml

Конфигурация:
flip_idx:
- 0
- 1
- 2
- 3
kpt_shape:
- 4
- 3
names:
- Chess-Board
nc: 1
path: /content/chess_board_corners_yolo11/dataset
test: /content/chess_board_corners_yolo11/dataset/test/images
train: /content/chess_board_corners_yolo11/dataset/train/images
val: /content/chess_board_corners_yolo11/dataset/valid/images


✓ YAML файл готов для keypoint detection


## Проверка доступности GPU

In [6]:
# Проверка доступности GPU
def check_gpu_availability():
    """Проверка доступности GPU и вывод информации"""
    print("=" * 60)
    print("ПРОВЕРКА ДОСТУПНОСТИ GPU")
    print("=" * 60)

    cuda_available = torch.cuda.is_available()
    print(f"CUDA доступна: {cuda_available}")

    if cuda_available:
        print(f"Количество GPU: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"\nGPU {i}:")
            print(f"  Название: {torch.cuda.get_device_name(i)}")
            print(f"  Память: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

        current_device = torch.cuda.current_device()
        print(f"\nТекущее устройство: GPU {current_device}")
        print(f"Имя устройства: {torch.cuda.get_device_name(current_device)}")

        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"\nИспользование памяти GPU:")
        print(f"  Выделено: {allocated:.2f} GB")
        print(f"  Зарезервировано: {reserved:.2f} GB")

        return '0'
    else:
        print("\n⚠ ВНИМАНИЕ: GPU не доступна!")
        print("Обучение будет выполняться на CPU (очень медленно)")
        print("\nДля Colab:")
        print("1. Runtime → Change runtime type")
        print("2. Hardware accelerator → GPU")
        print("3. Сохраните и перезапустите runtime")
        return 'cpu'

DEVICE = check_gpu_availability()
print(f"\n{'='*60}")
print(f"Будет использовано устройство: {DEVICE}")
print(f"{'='*60}\n")

ПРОВЕРКА ДОСТУПНОСТИ GPU
CUDA доступна: True
Количество GPU: 1

GPU 0:
  Название: Tesla T4
  Память: 14.74 GB

Текущее устройство: GPU 0
Имя устройства: Tesla T4

Использование памяти GPU:
  Выделено: 0.00 GB
  Зарезервировано: 0.00 GB

Будет использовано устройство: 0



## Загрузка предобученной модели YOLO11s

In [7]:
# Загрузка предобученной модели YOLO11s для keypoint detection
# yolo11s-pose.pt - это предобученная модель специально для pose/keypoint detection
# Если её нет, используем обычную yolo11s.pt и переключаем на task='pose'
model_name = f'yolo11{MODEL_SIZE}-pose.pt'  # pose модель для keypoint detection

try:
    model = YOLO(model_name)
    print(f"✓ Модель загружена: {model_name}")
    print("   Это предобученная модель для pose/keypoint detection")
except Exception as e:
    print(f"⚠ Не удалось загрузить {model_name}, пробуем обычную модель...")
    print(f"   Ошибка: {e}")
    # Если pose модель недоступна, загружаем обычную и переключаем на pose task
    model = YOLO(f'yolo11{MODEL_SIZE}.pt')
    print(f"✓ Модель загружена: yolo11{MODEL_SIZE}.pt")
    print("   Будет использован режим pose для keypoint detection (task='pose')")

✓ Модель загружена: yolo11s-pose.pt
   Это предобученная модель для pose/keypoint detection


## Обучение модели

In [8]:
# Обучение модели для keypoint detection
print("Начинаем обучение модели для keypoint detection...")
print(f"Конфигурация датасета: {dataset_yaml}")
print(f"Устройство для обучения: {DEVICE}")
print(f"Размер изображения: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Keypoints: {NUM_KEYPOINTS}")

# Параметры обучения для keypoint detection
results = model.train(
    data=str(dataset_yaml),
    task='pose',  # Режим pose для keypoint detection
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    device=DEVICE,
    patience=PATIENCE,
    project=str(RESULTS_DIR),
    name='chess_board_corners',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    seed=42,
    deterministic=True,
    single_cls=False,
    rect=False,
    cos_lr=False,
    close_mosaic=10,
    resume=False,
    amp=True,  # Automatic Mixed Precision
    fraction=1.0,
    profile=False,
    freeze=None,
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    pose=12.0,  # Weight для keypoint loss
    kobj=1.0,  # Keypoint object loss weight
    label_smoothing=0.0,
    nbs=64,
    overlap_mask=True,
    mask_ratio=4,
    dropout=0.0,
    val=True,
    plots=True,
    save=True,
    save_period=10,
    workers=8,
)

print("\n" + "="*60)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*60)
print(f"Результаты сохранены в: {RESULTS_DIR / 'chess_board_corners'}")
print(f"Лучшая модель: {RESULTS_DIR / 'chess_board_corners' / 'weights' / 'best.pt'}")

Начинаем обучение модели для keypoint detection...
Конфигурация датасета: /content/chess_board_corners_yolo11/dataset/data.yaml
Устройство для обучения: 0
Размер изображения: 1280x1280
Batch size: 8
Epochs: 100
Keypoints: 4
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/chess_board_corners_yolo11/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou

## Валидация модели

In [9]:
# Валидация обученной модели
best_model_path = RESULTS_DIR / 'chess_board_corners' / 'weights' / 'best.pt'

if best_model_path.exists():
    print(f"Загрузка лучшей модели: {best_model_path}")
    trained_model = YOLO(str(best_model_path))

    # Валидация на тестовом наборе
    print("\nЗапуск валидации...")
    val_results = trained_model.val(
        data=str(dataset_yaml),
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        plots=True,
        save_json=False,
    )

    print("\n" + "="*60)
    print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ")
    print("="*60)
    if hasattr(val_results, 'results_dict'):
        print(f"mAP50: {val_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
        print(f"mAP50-95: {val_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
        if 'metrics/pose/mAP50' in val_results.results_dict:
            print(f"Keypoint mAP50: {val_results.results_dict.get('metrics/pose/mAP50', 'N/A')}")
            print(f"Keypoint mAP50-95: {val_results.results_dict.get('metrics/pose/mAP50-95', 'N/A')}")
else:
    print("⚠ Лучшая модель не найдена. Проверьте результаты обучения.")

Загрузка лучшей модели: /content/chess_board_corners_yolo11/results/chess_board_corners/weights/best.pt

Запуск валидации...
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLO11s-pose summary (fused): 109 layers, 9,700,263 parameters, 0 gradients, 22.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3108.4±367.3 MB/s, size: 268.2 KB)
val: Scanning /content/chess_board_corners_yolo11/dataset/valid/labels.cache... 12 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 12/12 21.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.4s
                   all         12         12          1          1      0.995      0.995          0          0          0          0
Speed: 23.5ms preprocess, 54.5ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to /content/runs/pose/val

РЕЗУЛЬТАТЫ ВАЛИДАЦИИ
mAP50: 0.995
m

## Тестирование инференса на примерах

In [10]:
# Тестирование на нескольких изображениях
best_model_path = RESULTS_DIR / 'chess_board_corners' / 'weights' / 'best.pt'

if best_model_path.exists():
    trained_model = YOLO(str(best_model_path))

    # Берем несколько тестовых изображений
    test_images_dir = DATASET_DIR / 'test' / 'images'
    if not test_images_dir.exists():
        test_images_dir = DATASET_DIR / 'valid' / 'images'

    if test_images_dir.exists():
        test_images = list(test_images_dir.glob('*.jpg'))[:5]  # Первые 5 изображений

        print(f"Тестирование на {len(test_images)} изображениях...")

        for img_path in test_images:
            print(f"\nОбработка: {img_path.name}")

            # Предсказание
            results = trained_model.predict(
                source=str(img_path),
                imgsz=IMG_SIZE,
                conf=0.25,
                device=DEVICE,
                save=True,
                save_txt=False,
            )

            # Вывод результатов
            for result in results:
                if result.keypoints is not None and len(result.keypoints.data) > 0:
                    kpts = result.keypoints.data[0].cpu().numpy()  # Первый объект
                    print(f"  Найдено keypoints: {len(kpts)} точек")
                    for i, kpt in enumerate(kpts):
                        if len(kpt) >= 2:
                            x, y = kpt[0], kpt[1]
                            vis = kpt[2] if len(kpt) > 2 else 1.0
                            print(f"    {KEYPOINT_NAMES[i] if i < len(KEYPOINT_NAMES) else f'point_{i}'}: ({x:.1f}, {y:.1f}), visibility={vis:.2f}")
                else:
                    print(f"  ⚠ Keypoints не найдены")

        print(f"\n✓ Результаты сохранены в: {RESULTS_DIR / 'chess_board_corners' / 'predict'}")
    else:
        print("⚠ Тестовая директория не найдена")
else:
    print("⚠ Модель не найдена. Сначала обучите модель.")

Тестирование на 5 изображениях...

Обработка: IMG_4630_JPG.rf.0bcde6ae270fc56d840f9f6fe1b97444.jpg

image 1/1 /content/chess_board_corners_yolo11/dataset/test/images/IMG_4630_JPG.rf.0bcde6ae270fc56d840f9f6fe1b97444.jpg: 1280x960 1 Chess-Board, 65.0ms
Speed: 8.6ms preprocess, 65.0ms inference, 2.1ms postprocess per image at shape (1, 3, 1280, 960)
Results saved to /content/runs/pose/predict
  Найдено keypoints: 4 точек
    corner_a1: (433.5, 1053.0), visibility=0.99
    corner_h1: (472.5, 598.8), visibility=0.99
    corner_h8: (568.5, 584.8), visibility=0.97
    corner_a8: (462.6, 971.6), visibility=0.98

Обработка: IMG_4587_JPG.rf.6b3ad3b31dc0e1b785d0c0b4cf5c12be.jpg

image 1/1 /content/chess_board_corners_yolo11/dataset/test/images/IMG_4587_JPG.rf.6b3ad3b31dc0e1b785d0c0b4cf5c12be.jpg: 1280x960 1 Chess-Board, 36.6ms
Speed: 7.5ms preprocess, 36.6ms inference, 1.5ms postprocess per image at shape (1, 3, 1280, 960)
Results saved to /content/runs/pose/predict
  Найдено keypoints: 4 точек
 

## Скачивание обученной модели

In [ ]:
# Подготовка модели для скачивания
best_model_path = RESULTS_DIR / 'chess_board_corners' / 'weights' / 'best.pt'
last_model_path = RESULTS_DIR / 'chess_board_corners' / 'weights' / 'last.pt'

if best_model_path.exists():
    print(f"✓ Лучшая модель: {best_model_path}")
    print(f"  Размер: {best_model_path.stat().st_size / 1024 / 1024:.2f} MB")
    print(f"\nДля скачивания в Colab используйте:")
    print(f"  from google.colab import files")
    print(f"  files.download('{best_model_path}')")

    # Автоматическое скачивание
    try:
        from google.colab import files
        print(f"\nСкачивание лучшей модели...")
        files.download(str(best_model_path))
        print("✓ Модель скачана")
    except ImportError:
        print("\n⚠ google.colab не доступен (не в Colab?)")
        print(f"Скопируйте модель вручную: {best_model_path}")
else:
    print("⚠ Лучшая модель не найдена")
    if last_model_path.exists():
        print(f"Используется последняя модель: {last_model_path}")
        try:
            from google.colab import files
            files.download(str(last_model_path))
        except ImportError:
            print(f"Скопируйте модель вручную: {last_model_path}")